In [1]:
import numpy as np
np.bool = bool
import matplotlib.pyplot as plt

from wzk import sql2, trajectory

# Create a dataloader for the paths

In [2]:
file = './SingleSphere02_one-world.db'
# TODO change to you own file path


n_voxels = 64
voxel_size = 10 / 64     # in m
extent = [0, 10, 0, 10]  # in m
n_waypoints = 20  # start + 20 inner points + end
n_dim = 2
n_paths_per_world = 1000
n_worlds = 5000


worlds = sql2.get_values_sql(file=file, table="worlds", values_only=False)
print(worlds.head())

   world_i32                                            img_cmp
0          0  b'x\xda\xcd\x97I\x16\xc20\x0cC\xad\xfb_\x9a\x0...
1          1  b"x\xda\xed\x97\xcd\x12\xc2 \x0c\x84\xb3\xef\x...
2          2  b'x\xda\xed\x97\xc1\x0e\xc30\x08C\xf1\xff\xff\...
3          3  b"x\xda\xe5\x961\x16\xc30\x08C\xad\xfb_\xbaC\x...
4          4  b'x\xda\xc5\x96A\x12\x850\x08C\xc9\xfd/\xed\xc...


In [3]:
path_idx_for_whole_dataset = np.arange(1000000)
paths = sql2.get_values_sql(file=file, table='paths', rows=path_idx_for_whole_dataset, values_only=False)
paths.head()

,world_i32,sample_i32,q_f32,objective_f32,feasible_b
0,0,0,"[0.16710596, 4.7555532, 0.4675184, 4.849919, 0...",0.415048,1
1,0,1,"[0.16710907, 8.536185, 0.471434, 8.432489, 0.7...",0.460750,1
2,0,1,"[0.16710907, 8.536185, 0.8752723, 8.414336, 1....",2.132938,1
3,0,1,"[0.16710907, 8.536185, 0.53698635, 8.336125, 0...",0.584670,1
4,0,1,"[0.16710907, 8.536185, 0.61476713, 8.472488, 1...",0.665006,1


In [4]:
paths.describe()

,world_i32,sample_i32,objective_f32,feasible_b
count,509278.0,509278.000000,509278.000000,509278.0
mean,0.0,50300.161652,1.704607,1.0
std,0.0,28935.349994,1.166201,0.0
min,0.0,0.000000,0.006600,1.0
25%,0.0,24801.000000,0.852067,1.0
50%,0.0,51491.500000,1.424609,1.0
75%,0.0,75189.000000,2.291609,1.0
max,0.0,100030.000000,11.172982,1.0


In [5]:
paths = paths[paths['world_i32'] == 0] # just take the first world
paths.describe()

,world_i32,sample_i32,objective_f32,feasible_b
count,509278.0,509278.000000,509278.000000,509278.0
mean,0.0,50300.161652,1.704607,1.0
std,0.0,28935.349994,1.166201,0.0
min,0.0,0.000000,0.006600,1.0
25%,0.0,24801.000000,0.852067,1.0
50%,0.0,51491.500000,1.424609,1.0
75%,0.0,75189.000000,2.291609,1.0
max,0.0,100030.000000,11.172982,1.0


torch.Size([20, 2])

In [22]:
from models import TemporalUnet

In [23]:
# define the model
model = TemporalUnet(horizon=20, transition_dim=2, cond_dim=2, dim=32, dim_mults=(1, 2, 4, 8), attention=False)

[ models/temporal ] Channel dimensions: [(2, 32), (32, 64), (64, 128), (128, 256)]
[(2, 32), (32, 64), (64, 128), (128, 256)]
